# Random Forest

In [1]:
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio import features
import numpy as np
import glob
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression

## Prepare data

In [2]:
# import study area
study_area = rxr.open_rasterio('output-data/study-area/study_area_kalimantan.tif').squeeze()

In [7]:
# import predictor layers (squeeze removed band dimension)

drainage_dist = rxr.open_rasterio("output-data/predictor-layers/drainage-dist.tif").squeeze()
forest_dist = rxr.open_rasterio("output-data/predictor-layers/forest-dist.tif").squeeze()
pre_fire_evi_2m = rxr.open_rasterio("output-data/predictor-layers/evi_2month_pre.tif").squeeze()
pre_fire_evi_2y = rxr.open_rasterio("output-data/predictor-layers/evi_2year_pre.tif").squeeze()
post_fire_evi = rxr.open_rasterio('output-data/predictor-layers/post_fire_evi.tif').squeeze()
burn_severity = rxr.open_rasterio("output-data/predictor-layers/reprojected_Burn_Severity.tif").squeeze()
aridity_index = rxr.open_rasterio('output-data/predictor-layers/aridity_index.tif').squeeze()
fire_count = rxr.open_rasterio('output-data/predictor-layers/fire_count.tif').squeeze()
pre_fire_nbr = rxr.open_rasterio('output-data/predictor-layers/reprojected_NBR_PreFire_Kalimantan.tif').squeeze()
post_fire_nbr = rxr.open_rasterio('output-data/predictor-layers/reprojected_NBR_PostFire_Kalimantan.tif').squeeze()
dnbr = pre_fire_nbr - post_fire_nbr

In [11]:
# import vegetation metrics
# I guess i renamed them here as outcome layers, in my folder, i hope is not so confusing
evi_recovery_2y = rxr.open_rasterio("output-data/EVI/dif_evi_short_2017_minus_2015.tif").squeeze()
evi_recovery_5y = rxr.open_rasterio("output-data/EVI/dif_evi_long_2020_minus_2015.tif").squeeze()

In [13]:
# combine in one dataset
ds = xr.Dataset({
    "study_area": study_area,
    "drainage_dist": drainage_dist,
    "forest_dist": forest_dist,
    "pre_fire_evi_2m": pre_fire_evi_2m,
    "pre_fire_evi_2y": pre_fire_evi_2y,
    'post_fire_evi': post_fire_evi,
    "burn_severity": burn_severity,
    'aridity_index': aridity_index,
    'fire_count': fire_count,
    'pre_fire_nbr': pre_fire_nbr,
    'post_fire_nbr': post_fire_nbr,
    'dnbr': dnbr,
    'recovery_2y': evi_recovery_2y,
    'recovery_5y': evi_recovery_5y
})

In [14]:
# transform to pandas dataframe (only study area pixels)
df = ds.to_dataframe().reset_index()

# remove pixels outside study area
df = df[df.study_area == 1]

### Explore variable correlation and data availablity

In [17]:
df.corr()

,x,y,band,spatial_ref,study_area,drainage_dist,forest_dist,pre_fire_evi_2m,pre_fire_evi_2y,post_fire_evi,burn_severity,aridity_index,fire_count,pre_fire_nbr,post_fire_nbr,dnbr,recovery_2y,recovery_5y
x,1.000000,0.394010,NaN,NaN,NaN,-0.003146,-0.016045,-0.067628,-0.080815,0.075780,0.060688,-0.487716,-0.032096,-0.000775,-0.161570,0.100795,-0.009324,-0.084138
y,0.394010,1.000000,NaN,NaN,NaN,-0.084022,-0.045086,0.078528,0.119924,0.176324,-0.069370,0.049549,-0.019377,-0.016420,0.005543,-0.016259,0.021370,0.013929
band,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spatial_ref,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
drainage_dist,-0.003146,-0.084022,NaN,NaN,NaN,1.000000,0.031360,0.059825,0.068905,-0.054708,-0.027620,0.104451,-0.062167,0.000006,0.104131,-0.065346,0.055793,-0.001126
forest_dist,-0.016045,-0.045086,NaN,NaN,NaN,0.031360,1.000000,0.154499,0.180773,-0.060761,0.403209,-0.114850,0.083917,0.160501,-0.101484,0.188615,0.119813,0.179292
pre_fire_evi_2m,-0.067628,0.078528,NaN,NaN,NaN,0.059825,0.154499,1.000000,0.856164,0.164742,0.277634,-0.135415,0.027636,0.598900,-0.073013,0.504780,0.368166,0.492809
pre_fire_evi_2y,-0.080815,0.119924,NaN,NaN,NaN,0.068905,0.180773,0.856164,1.000000,0.170412,0.218947,-0.111607,0.021526,0.453751,-0.063019,0.392512,0.402967,0.529924
post_fire_evi,0.075780,0.176324,NaN,NaN,NaN,-0.054708,-0.060761,0.164742,0.170412,1.000000,-0.275208,-0.062859,0.102754,0.078972,0.693911,-0.416101,-0.609992,-0.584426


In [16]:
df.isnull().sum()

x                      0
y                      0
band                   0
spatial_ref            0
study_area             0
drainage_dist          0
forest_dist            0
pre_fire_evi_2m    21158
pre_fire_evi_2y     9357
post_fire_evi      34791
burn_severity          0
aridity_index          0
fire_count             0
pre_fire_nbr       21348
post_fire_nbr      21348
dnbr               21348
recovery_2y        42121
recovery_5y        42019
dtype: int64

### Clean

In [22]:
# drop unnecessary columns
df =  df.drop(['x', 'y', 'band', 'spatial_ref', 'study_area'], axis='columns')

# remove all rows with at least one nan
df_clean = df.dropna()

In [23]:
df_clean

,drainage_dist,forest_dist,pre_fire_evi_2m,pre_fire_evi_2y,post_fire_evi,burn_severity,aridity_index,fire_count,pre_fire_nbr,post_fire_nbr,dnbr,recovery_2y,recovery_5y
2111942,46.830687,192.917760,0.494554,0.483023,0.298354,2,2.240730,5,0.526129,0.340228,0.185901,0.185893,0.088566
2115764,77.790595,271.987883,0.462922,0.368601,0.224532,2,2.241955,5,0.479982,0.290679,0.189303,0.192333,0.131759
2115767,109.970762,444.076474,0.479131,0.338610,0.377789,1,2.241169,5,0.525499,0.445667,0.079832,0.174834,0.175769
2150326,971.022014,252.670193,0.273366,0.237709,0.227097,1,2.107037,0,0.357129,0.324605,0.032524,-0.028904,-0.128537
2150327,1130.598930,443.707282,0.339429,0.269376,0.310728,1,2.110589,0,0.359711,0.383961,-0.024250,-0.007325,-0.168507
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15330873,269.320351,480.221873,0.550507,0.525888,0.147638,3,1.434707,0,0.533267,0.201099,0.332168,0.394073,0.462580
15330874,229.881356,422.639693,0.555042,0.543022,0.131511,3,1.433729,0,0.520513,0.213463,0.307050,0.438983,0.451797
15330875,396.803480,365.717793,0.555286,0.553890,0.142212,3,1.432751,0,0.600074,0.192311,0.407763,0.481298,0.456576
15334693,424.448586,328.792851,0.561896,0.531324,0.133935,3,1.436221,0,0.537362,0.276312,0.261050,0.430574,0.471835


In [27]:
predictor_cols = df_clean.columns.to_list()[:-2]
predictor_cols

['drainage_dist',
 'forest_dist',
 'pre_fire_evi_2m',
 'pre_fire_evi_2y',
 'post_fire_evi',
 'burn_severity',
 'aridity_index',
 'fire_count',
 'pre_fire_nbr',
 'post_fire_nbr',
 'dnbr']

In [ ]:
# clean data (remove impossible values 

# remove missing data

In [29]:
# prepare 10-fold CV
from sklearn.model_selection import KFold

# Set up 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=123)

# Prepare X and y for both models
X_2y = df_clean[predictor_cols]
y_2y = df_clean["recovery_2y"]

X_5y = df_clean[predictor_cols]
y_5y = df_clean["recovery_5y"]

print(f"2-year recovery: {X_2y.shape[0]} samples, {X_2y.shape[1]} features")
print(f"5-year recovery: {X_5y.shape[0]} samples, {X_5y.shape[1]} features")

2-year recovery: 5727 samples, 11 features
5-year recovery: 5727 samples, 11 features


## Fit model

In [30]:
# set up model
rf = RandomForestRegressor(
    n_estimators=500, # number of trees
    random_state=123 # for reproducibility
    )

In [31]:
## Fit model
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# set up models
rf_2y = RandomForestRegressor(
    n_estimators=500,
    random_state=123,
    n_jobs=-1  # use all CPU cores
)

rf_5y = RandomForestRegressor(
    n_estimators=500,
    random_state=123,
    n_jobs=-1
)

# Store CV results
cv_results_2y = {"r2": [], "rmse": [], "mae": []}
cv_results_5y = {"r2": [], "rmse": [], "mae": []}

# 10-fold CV for 2-year model
print("\n=== 2-Year Recovery Model ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X_2y), 1):
    X_train, X_test = X_2y.iloc[train_idx], X_2y.iloc[test_idx]
    y_train, y_test = y_2y.iloc[train_idx], y_2y.iloc[test_idx]
    
    rf_2y.fit(X_train, y_train)
    y_pred = rf_2y.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    cv_results_2y["r2"].append(r2)
    cv_results_2y["rmse"].append(rmse)
    cv_results_2y["mae"].append(mae)
    
    print(f"Fold {fold}: R² = {r2:.3f}, RMSE = {rmse:.4f}, MAE = {mae:.4f}")

print(f"\nMean R² = {np.mean(cv_results_2y['r2']):.3f} ± {np.std(cv_results_2y['r2']):.3f}")
print(f"Mean RMSE = {np.mean(cv_results_2y['rmse']):.4f} ± {np.std(cv_results_2y['rmse']):.4f}")

# 10-fold CV for 5-year model
print("\n=== 5-Year Recovery Model ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X_5y), 1):
    X_train, X_test = X_5y.iloc[train_idx], X_5y.iloc[test_idx]
    y_train, y_test = y_5y.iloc[train_idx], y_5y.iloc[test_idx]
    
    rf_5y.fit(X_train, y_train)
    y_pred = rf_5y.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    cv_results_5y["r2"].append(r2)
    cv_results_5y["rmse"].append(rmse)
    cv_results_5y["mae"].append(mae)
    
    print(f"Fold {fold}: R² = {r2:.3f}, RMSE = {rmse:.4f}, MAE = {mae:.4f}")

print(f"\nMean R² = {np.mean(cv_results_5y['r2']):.3f} ± {np.std(cv_results_5y['r2']):.3f}")
print(f"Mean RMSE = {np.mean(cv_results_5y['rmse']):.4f} ± {np.std(cv_results_5y['rmse']):.4f}")

# Fit final models on full data
rf_2y.fit(X_2y, y_2y)
rf_5y.fit(X_5y, y_5y)


=== 2-Year Recovery Model ===
Fold 1: R² = 0.700, RMSE = 0.0693, MAE = 0.0389
Fold 2: R² = 0.804, RMSE = 0.0535, MAE = 0.0365
Fold 3: R² = 0.610, RMSE = 0.0880, MAE = 0.0406
Fold 4: R² = 0.737, RMSE = 0.0660, MAE = 0.0367
Fold 5: R² = 0.756, RMSE = 0.0644, MAE = 0.0402
Fold 6: R² = 0.758, RMSE = 0.0581, MAE = 0.0373
Fold 7: R² = 0.655, RMSE = 0.0764, MAE = 0.0414
Fold 8: R² = 0.829, RMSE = 0.0521, MAE = 0.0354
Fold 9: R² = 0.702, RMSE = 0.0717, MAE = 0.0412
Fold 10: R² = 0.765, RMSE = 0.0600, MAE = 0.0389

Mean R² = 0.732 ± 0.063
Mean RMSE = 0.0659 ± 0.0104

=== 5-Year Recovery Model ===
Fold 1: R² = 0.831, RMSE = 0.0537, MAE = 0.0397
Fold 2: R² = 0.829, RMSE = 0.0539, MAE = 0.0385
Fold 3: R² = 0.829, RMSE = 0.0548, MAE = 0.0388
Fold 4: R² = 0.838, RMSE = 0.0512, MAE = 0.0378
Fold 5: R² = 0.819, RMSE = 0.0542, MAE = 0.0373
Fold 6: R² = 0.817, RMSE = 0.0526, MAE = 0.0383
Fold 7: R² = 0.828, RMSE = 0.0531, MAE = 0.0398
Fold 8: R² = 0.845, RMSE = 0.0501, MAE = 0.0364
Fold 9: R² = 0.850, 

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

Your Random Forest results show **moderate but acceptable performance** for ecological modeling:

## **2-Year Recovery Model:**
- **R² = 0.351** → Explains ~35% of variance
- **RMSE = 0.102** → Average error ±0.10 EVI units
- **MAE = 0.072** → Typical error ~0.07 EVI units
- **CV stability**: std = 0.031 (reasonable)

**Assessment**: **Fair to moderate**. The model captures some patterns but misses ~65% of variance.

## **5-Year Recovery Model:**
- **R² = 0.469** → Explains ~47% of variance
- **RMSE = 0.093** → Average error ±0.09 EVI units  
- **MAE = 0.069** → Typical error ~0.07 EVI units
- **CV stability**: std = 0.036 (good)

**Assessment**: **Moderate**. Better than 2-year model, captures nearly half the variance.

---

## **Interpretation:**

✅ **What's working:**
- 5-year model > 2-year model (longer recovery is more predictable)
- Low CV variance → stable, not overfitting
- RMSE/MAE reasonable for EVI difference scale

⚠️ **What's limiting:**
- R² < 0.5 suggests **high unexplained variance**
- Likely missing key factors:
  - **Weather/rainfall** (not included)
  - **Soil conditions** 
  - **Fire intensity details**
  - **Species composition**
  - **Local management**

---

## **Next steps to improve:**

````python
## Model accuracy
import matplotlib.pyplot as plt

# Plot CV results comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ["r2", "rmse", "mae"]
titles = ["R² Score", "RMSE", "MAE"]

for ax, metric, title in zip(axes, metrics, titles):
    ax.boxplot([cv_results_2y[metric], cv_results_5y[metric]], labels=["2-year", "5-year"])
    ax.set_title(title)
    ax.set_ylabel(title)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Check feature importance
print("\n=== Feature Importance ===")
feat_imp_2y = pd.DataFrame({
    "feature": predictor_cols,
    "importance_2y": rf_2y.feature_importances_
}).sort_values("importance_2y", ascending=False)

feat_imp_5y = pd.DataFrame({
    "feature": predictor_cols,
    "importance_5y": rf_5y.feature_importances_
}).sort_values("importance_5y", ascending=False)

print("\n2-Year Model:")
print(feat_imp_2y)
print("\n5-Year Model:")
print(feat_imp_5y)
````

**Your models are usable** but explain <50% variance—typical for complex ecological recovery where many unmeasured factors matter.

## Model accuracy

## Variable importance

In [32]:
import pandas as pd

# 2-year model importance
importance_2y = pd.DataFrame({
    "variable": X_2y.columns,
    "importance": rf_2y.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_2y)

           variable  importance
4     post_fire_evi    0.449550
3   pre_fire_evi_2y    0.263126
6     aridity_index    0.074737
2   pre_fire_evi_2m    0.041119
0     drainage_dist    0.038322
1       forest_dist    0.031804
9     post_fire_nbr    0.029464
8      pre_fire_nbr    0.026633
10             dnbr    0.025354
7        fire_count    0.016972
5     burn_severity    0.002919


In [33]:
importance_5y = pd.DataFrame({
    "variable": X_5y.columns,
    "importance": rf_5y.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_5y)

           variable  importance
4     post_fire_evi    0.413779
3   pre_fire_evi_2y    0.387065
6     aridity_index    0.034325
2   pre_fire_evi_2m    0.031913
0     drainage_dist    0.030728
9     post_fire_nbr    0.025001
1       forest_dist    0.022806
8      pre_fire_nbr    0.021026
10             dnbr    0.017585
7        fire_count    0.014087
5     burn_severity    0.001686


## Partial dependence plots